## Table 5 (dna): Lineage-Holdout vs. Random-Split Significance Test

Tests whether each architecture's mean AUC under lineage holdout differs significantly from its random-split test AUC, per architecture, paired across the 9 eligible drugs, with a Holm-Bonferroni correction across the 4 architectures.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

ROOT = Path("../results/test_output_summary/task1")

SOURCES = {
    "SD-LogReg": {
        "random": ROOT / "random_split/random_split_summary_logreg.csv",
        "lineage": ROOT / "lineage_split/all_lineage_summary_logreg.csv",
    },
    "SD-CNN": {
        "random": ROOT / "random_split/random_split_summary_sdcnn.csv",
        "lineage": ROOT / "lineage_split/all_lineage_summary_sdcnn.csv",
    },
    "Evo2-CNN-zero-shot": {
        "random": ROOT / "random_split/random_split_summary_evo2_token.csv",
        "lineage": ROOT / "lineage_split/all_lineage_summary_evo2_zeroshot.csv",
    },
    "Evo2-CNN-pca10": {
        "random": ROOT / "random_split/random_split_summary_evo2_pca.csv",
        "lineage": ROOT / "lineage_split/all_lineage_summary_evo2_pca.csv",
    },
}

DRUGS = [
    "rifampicin", "isoniazid", "ethambutol", "pyrazinamide",
    "streptomycin", "moxifloxacin", "ethionamide", "amikacin",
    "kanamycin", "capreomycin",
]


def normalize(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def find_drug_column(df):
    expected = {normalize(drug) for drug in DRUGS}
    scores = {
        col: df[col].dropna().map(normalize).isin(expected).sum()
        for col in df.columns
    }

    best_col = max(scores, key=scores.get)
    if scores[best_col] > 0:
        return best_col

    named_cols = [col for col in df.columns if "drug" in str(col).lower()]
    if named_cols:
        return named_cols[0]

    raise ValueError(f"Could not identify drug column. Columns: {list(df.columns)}")


def get_auc_columns(df):
    auc_cols = [
        col for col in df.columns
        if "auc" in normalize(col)
        and not any(term in normalize(col) for term in ("aucpr", "std", "sem"))
    ]

    # Use a summarized mean AUC column if the file provides one.
    mean_cols = [
        col for col in auc_cols
        if "mean" in normalize(col) or "average" in normalize(col)
    ]
    if mean_cols:
        return [mean_cols[0]]

    if auc_cols:
        return auc_cols

    plain_mean_cols = [
        col for col in df.columns
        if normalize(col) in {"mean", "average"}
    ]
    if plain_mean_cols:
        return [plain_mean_cols[0]]

    raise ValueError(f"Could not identify AUC column. Columns: {list(df.columns)}")


def load_auc_per_drug(path, lineage=False):
    df = pd.read_csv(path)

    if lineage and "feasible" in df.columns:
        feasible = df["feasible"].astype(str).str.lower().isin(
            {"true", "1", "yes"}
        )
        df = df.loc[feasible].copy()

    drug_col = find_drug_column(df)
    auc_cols = get_auc_columns(df)

    df["_drug"] = df[drug_col].map(normalize)
    df["_auc"] = df[auc_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    df = df.dropna(subset=["_auc"])

    # Average multiple rows, folds, or lineages into one AUC per drug.
    return df.groupby("_drug")["_auc"].mean()

In [ ]:
random_auc = {}
lineage_auc = {}

for architecture, paths in SOURCES.items():
    random_auc[architecture] = load_auc_per_drug(paths["random"])
    lineage_auc[architecture] = load_auc_per_drug(paths["lineage"], lineage=True)

# Compare the same eligible drugs across every architecture and split.
eligible_keys = {normalize(drug) for drug in DRUGS}
for values in [*random_auc.values(), *lineage_auc.values()]:
    eligible_keys &= set(values.index)

eligible_drugs = [
    normalize(drug) for drug in DRUGS
    if normalize(drug) in eligible_keys
]
if not eligible_drugs:
    raise ValueError("No drugs are present in every input file.")

excluded = [
    drug for drug in DRUGS
    if normalize(drug) not in eligible_keys
]
if excluded:
    print("Excluded drugs missing from at least one input:", ", ".join(excluded))

rows = []
for architecture in SOURCES:
    random_values = random_auc[architecture].loc[eligible_drugs].to_numpy()
    lineage_values = lineage_auc[architecture].loc[eligible_drugs].to_numpy()
    delta = lineage_values - random_values

    p_value = (
        1.0 if np.allclose(delta, 0)
        else wilcoxon(
            lineage_values, random_values, alternative="two-sided"
        ).pvalue
    )

    rows.append({
        "architecture": architecture,
        "mean_random_split_auc": random_values.mean(),
        "mean_lineage_holdout_auc": lineage_values.mean(),
        "mean_delta_auc": delta.mean(),
        "p_value": p_value,
        "n_drugs": len(eligible_drugs),
    })

result = pd.DataFrame(rows)
result["p_value_holm"] = multipletests(result["p_value"], method="holm")[1]

output_path = Path(
    "../results/significance_testing/table17_lineage_vs_random_significance_holm.csv"
)
result.to_csv(output_path, index=False)

print(f"[OK] Wrote {output_path}")
result.round(4)

[OK] Wrote table5_lineage_vs_random_significance_holm.csv


,architecture,mean_random_split_auc,mean_lineage_holdout_auc,mean_delta_auc,p_value,n_drugs,p_value_holm
0,SD-LogReg,0.8682,0.8322,-0.0359,0.0488,10,0.1465
1,SD-CNN,0.8829,0.8347,-0.0482,0.0645,10,0.1465
2,Evo2-CNN-zero-shot,0.8700,0.8238,-0.0462,0.0059,10,0.0234
3,Evo2-CNN-pca10,0.8625,0.8164,-0.0461,0.0645,10,0.1465
